In [ ]:
using IntervalArithmetic, LinearAlgebra, Serialization, SpecialFunctions, Arblib

In [ ]:
setprecision(8192*16)

In [ ]:
N = 50
N₀ = 50
cert = true
κ = interval(BigFloat, 4)
c⁻ = I"0.987"
c⁺ = I"1.177"

In [ ]:
setprecision(1024)

LinearAlgebra.norm(v::Vector) = sqrt(sum(v.^2))

function enlarge(x::Interval{BigFloat})
    isguaranteed(x) || error("interval is not guaranteed")
    return interval(BigFloat(inf(x), RoundDown), BigFloat(sup(x), RoundUp))
end

SpecialFunctions.besseli(ν::Interval{BigFloat},z::Interval{BigFloat}) = interval(besseli(Arb(ν), Arb(z)))
SpecialFunctions.besseli(ν::Rational,z::Interval{BigFloat}) = interval(besseli(interval(BigFloat, ν), z))

function compute_b1(κ)
    κ̂ = κ^2/interval(8)
    num = κ^2*(besseli(-1//4, κ̂)+besseli(3//4, κ̂)+besseli(5//4, κ̂))+(interval(4)+κ^2)*besseli(1//4, κ̂)
    denom = interval(2)*κ*(besseli(-1//4, κ̂)+besseli(1//4, κ̂))
    return num/denom
end

function Base.inv(P::Bidiagonal{Interval{Float64}, Vector{Interval{Float64}}})
    if P.uplo == 'U'
        C = -P.ev./P.dv[1:end-1]
        invC = UpperTriangular(zeros(eltype(P), size(P)))
        for i in 1:size(P)[1]
            invC[i,i:end] = cumprod([one(eltype(P)); C[i:end]])
        end
        return UpperTriangular(invC ./ P.dv')
    else
        return LowerTriangular(inv(P')')
    end
end    

function Base.:*(D::Diagonal, v::Vector)
    return diag(D).*v
end

function Base.:*(D::Diagonal, A::Matrix)
    return diag(D).*A
end

function Base.:*(A::Matrix, D::Diagonal)
    return A.*diag(D)'
end

# rigorous upper bound of the 2-norm of a matrix
function op_norm(A)
    if size(A) == (2,2)
        Z = sqrt(sum(A.^2) + sqrt(((A[1,2]+A[2,1])^2+(A[1,1]-A[2,2])^2)*((A[1,2]-A[2,1])^2+(A[1,1]+A[2,2])^2)))/sqrt(interval(2))
        if isguaranteed(Z)
            return interval(sup(Z))
        else
            return Z
        end
    else
        B = A'A
        Λ̄, V̄ = eigen(Symmetric(mid.(B)))
        Λ = inv(interval.(V̄))*B*interval.(V̄)
        all(isguaranteed.(Λ)) || error("matrix not guaranteed")
        σ̄ = sup(sqrt(maximum(abs.(diag(Λ) + interval(-1,1)*[sum(abs.(Λ[i,1:i-1]))+sum(abs.(Λ[i,i+1:end])) for i=1:size(B)[1]]))))
        σ̲ = inf(norm(A*interval.(V̄)[:,end])/norm(interval.(V̄)[:,end]))
        return interval(σ̲, σ̄)
    end
end

In [ ]:
setprecision(8192)
b₁ = compute_b1(κ)
C_α = interval(3)^interval(1//4)/sqrt(c⁺)
θ = c⁺^2*interval((1+1//N₀)*(1+2//N₀))^interval(1//4)/interval(3)
setprecision(precision(mid.(b₁)))
b₀ = interval(BigFloat, 0.0)
b = zeros(Interval{BigFloat},N+4)
b[1:2] = [b₀, b₁]
for k = 2:N+3
    b[k+1] = interval(4)+(interval(BigFloat, k-1)/b[k])-b[k]-b[k-1]
end
setprecision(1024)
b = b[2:end]
a = enlarge.(sqrt.(b))
b = enlarge.(b);

In [ ]:
α = interval.(Float64, interval.(collect(1:2:N+1))./a[1:2:N+1])
β = interval.(Float64, a[1:2:N+1].*a[2:2:N+2].*a[3:2:N+3])
A = Bidiagonal(α,  β[1:end-1], :U)
A⁻¹ = Matrix(interval.(Float64, inv(A)));

In [ ]:
C₁₂ = interval(1)/(C_α*sqrt(interval(1)-θ^2))*β[end]*norm(A⁻¹[:,end])
C₂₂ = interval(1)/(C_α*(interval(1)-θ));

In [ ]:
C₁ = norm([C₁₂, C₂₂]);

In [ ]:
α = interval.(Float64, interval.(collect(2:2:N))./a[2:2:N])
β = interval.(Float64, a[2:2:N].*a[3:2:N+1].*a[4:2:N+2])
A = Bidiagonal(α,  β[1:end-1], :U)
A⁻¹ = Matrix(interval.(Float64, inv(A)));

In [ ]:
C₁₂ = interval(1)/(C_α*sqrt(interval(1)-θ^2))*β[end]*norm(A⁻¹[:,end])
C₂₂ = interval(1)/(C_α*(interval(1)-θ));

In [ ]:
C₀ = norm([C₁₂, C₂₂]);

In [ ]:
C = max(C₁,C₀)

In [ ]:
sup(C)